In [20]:
import random
from collections import deque, defaultdict
from itertools import combinations

random.seed(42)

GRID_SIZE = 4
START_POSITION = (1, 1)
ACTIONS = ("fwd", "turn_l", "turn_r", "grab", "shoot", "climb")
DIRECTIONS = ("east", "north", "west", "south")
OFFSETS = {
    "east": (1, 0), "north": (0, 1),
    "west": (-1, 0), "south": (0, -1)
}
MAX_POPULATION_SIZE = 200
GA_THRESHOLD = 25
GAMMA = 0.95

MESSAGE_ORDER = (
    "breeze", "stench", "glitter", "bump", "scream", "at_start",
    "has_gold", "danger_ahead", "visited_here", "wumpus_near"
)


def inside(position):
    x, y = position
    return 1 <= x <= GRID_SIZE and 1 <= y <= GRID_SIZE


def adjacent(position):
    x, y = position
    return {
        (x + 1, y), (x - 1, y),
        (x, y + 1), (x, y - 1)
    } & {(px, py) for px in range(1, GRID_SIZE + 1)
         for py in range(1, GRID_SIZE + 1)}


def turn(direction, side):
    index = DIRECTIONS.index(direction)
    return DIRECTIONS[(index - 1 if side == "left" else index + 1) % 4]


def next_position(position, direction):
    dx, dy = OFFSETS[direction]
    return position[0] + dx, position[1] + dy


def shortest_path(start, targets, blocked=(), allowed=None):
    """Breadth-first shortest path to the nearest target position."""
    if start in targets:
        return []
    blocked = set(blocked)
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        position, path = queue.popleft()
        for neighbor in adjacent(position):
            if neighbor in visited or neighbor in blocked:
                continue
            if allowed is not None and neighbor not in allowed:
                continue
            new_path = path + [neighbor]
            if neighbor in targets:
                return new_path
            visited.add(neighbor)
            queue.append((neighbor, new_path))
    return None


def make_message_string(messages):
    return "".join("1" if messages[name] else "0" for name in MESSAGE_ORDER)


class World:
    def __init__(self, wumpus=None, gold=None, pits=None, pit_count=3):
        available = {
            (x, y) for x in range(1, GRID_SIZE + 1)
            for y in range(1, GRID_SIZE + 1)
        } - {START_POSITION}

        wumpus = wumpus or random.choice(tuple(available))
        if wumpus not in available:
            raise ValueError("Invalid Wumpus position")
        available.remove(wumpus)

        gold = gold or random.choice(tuple(available))
        if gold not in available:
            raise ValueError("Invalid gold position")
        available.remove(gold)

        pits = set(pits) if pits is not None else set(random.sample(tuple(available), pit_count))
        if not pits.issubset(available):
            raise ValueError("Invalid pit position")

        self.wumpus, self.gold, self.pits = wumpus, gold, pits
        self.agent_position = START_POSITION
        self.agent_direction = "east"
        self.arrow_available = True
        self.wumpus_alive = True
        self.has_gold = False
        self.agent_alive = True
        self.agent_escaped = False
        self.bump = False
        self.scream = False

    def get_percepts(self):
        here = self.agent_position
        near = adjacent(here)
        return {
            "breeze": bool(near & self.pits),
            "stench": self.wumpus_alive and (
                here == self.wumpus or self.wumpus in near
            ),
            "glitter": here == self.gold and not self.has_gold,
            "bump": self.bump,
            "scream": self.scream,
            "at_start": here == START_POSITION
        }

    def move_forward(self):
        self.bump = False
        self.scream = False
        destination = next_position(self.agent_position, self.agent_direction)
        if not inside(destination):
            self.bump = True
            return False
        self.agent_position = destination
        if destination in self.pits or (
            self.wumpus_alive and destination == self.wumpus
        ):
            self.agent_alive = False
        return True

    def turn_left(self):
        self.bump = False
        self.scream = False
        self.agent_direction = turn(self.agent_direction, "left")

    def turn_right(self):
        self.bump = False
        self.scream = False
        self.agent_direction = turn(self.agent_direction, "right")

    def grab(self):
        self.bump = False
        self.scream = False
        if self.agent_position == self.gold and not self.has_gold:
            self.has_gold = True
            return True
        return False

    def shoot(self):
        self.bump = False
        self.scream = False
        if not self.arrow_available:
            return False
        self.arrow_available = False
        position = self.agent_position
        while True:
            position = next_position(position, self.agent_direction)
            if not inside(position):
                return False
            if self.wumpus_alive and position == self.wumpus:
                self.wumpus_alive = False
                self.scream = True
                return True

    def climb(self):
        self.bump = False
        self.scream = False
        if self.agent_position == START_POSITION and self.has_gold:
            self.agent_escaped = True
            return True
        return False


class AgentKnowledge:
    def __init__(self):
        self.visited_positions = {START_POSITION}
        self.known_pits = set()
        self.known_wumpus = set()
        self.possible_pits = set()
        self.possible_wumpus = set()
        self.observations = []

    def update(self, position, percepts):
        self.visited_positions.add(position)
        self.observations.append((position, percepts["breeze"], percepts["stench"]))

        all_positions = {(x, y) for x in range(1, GRID_SIZE + 1)
                         for y in range(1, GRID_SIZE + 1)}

        breeze_neighborhoods = []
        no_breeze = set()
        for observed_position, breeze, stench in self.observations:
            neighborhood = adjacent(observed_position)
            if breeze:
                breeze_neighborhoods.append(neighborhood)
            else:
                no_breeze |= neighborhood

        pit_candidates = (set().union(*breeze_neighborhoods)
                          if breeze_neighborhoods else all_positions)
        pit_candidates -= no_breeze
        pit_candidates -= self.visited_positions
        self.possible_pits = pit_candidates
        self.known_pits = set(pit_candidates)

        stench_neighborhoods = []
        no_stench = set()
        for observed_position, breeze, stench in self.observations:
            neighborhood = adjacent(observed_position) | {observed_position}
            if stench:
                stench_neighborhoods.append(neighborhood)
            else:
                no_stench |= neighborhood

        wumpus_candidates = (set().union(*stench_neighborhoods)
                             if stench_neighborhoods else all_positions)
        wumpus_candidates -= no_stench
        wumpus_candidates -= self.visited_positions
        self.possible_wumpus = wumpus_candidates
        self.known_wumpus = set(wumpus_candidates)

    def update_after_shooting(self, world):
        if not world.wumpus_alive:
            self.known_wumpus.clear()
            self.possible_wumpus.clear()


def get_agent_messages(world, knowledge):
    messages = world.get_percepts()
    messages["has_gold"] = world.has_gold
    messages["visited_here"] = world.agent_position in knowledge.visited_positions
    messages["wumpus_near"] = bool(
        adjacent(world.agent_position) & knowledge.known_wumpus
    )
    ahead = next_position(world.agent_position, world.agent_direction)
    messages["danger_ahead"] = (
        ahead in knowledge.known_pits or ahead in knowledge.known_wumpus
    )
    return messages


class Classifier:
    def __init__(self, condition, action):
        self.condition = condition
        self.action = action
        self.prediction = 0.0
        self.error = 0.0
        self.fitness = 0.01
        self.experience = 0
        self.action_set_size = 1.0
        self.timestamp = 0

    def matches(self, message):
        return len(self.condition) == len(message) and all(
            c == "#" or c == m
            for c, m in zip(self.condition, message)
        )


class ClassifierSystem:
    def __init__(self):
        self.population = []
        self.time_step = 0

    def add_classifier(self, classifier):
        self.population.append(classifier)
        if len(self.population) > MAX_POPULATION_SIZE:
            self.delete_classifier()

    def delete_classifier(self):
        if not self.population:
            return
        self.population.remove(min(
            self.population,
            key=lambda c: c.fitness / (1 + c.experience)
        ))

    def matching(self, message):
        return [c for c in self.population if c.matches(message)]

    def get_or_create_match_set(self, message):
        match_set = self.matching(message)
        existing_actions = {c.action for c in match_set}
        covering = []
        for action in ACTIONS:
            if action not in existing_actions:
                condition = "".join("#" if random.random() < 0.30 else bit
                                    for bit in message)
                covering.append(Classifier(condition, action))
        self.population.extend(covering)
        while len(self.population) > MAX_POPULATION_SIZE:
            candidates = [c for c in self.population if c not in covering]
            if candidates:
                self.population.remove(min(
                    candidates,
                    key=lambda c: c.fitness / (1 + c.experience)
                ))
            else:
                break
        return self.matching(message)

    def action_set(self, match_set, action):
        return [c for c in match_set if c.action == action]

    def action_predictions(self, match_set):
        grouped = defaultdict(list)
        for classifier in match_set:
            grouped[classifier.action].append(classifier)
        predictions = {}
        for action, classifiers in grouped.items():
            total_fitness = sum(c.fitness for c in classifiers)
            if total_fitness:
                predictions[action] = sum(c.prediction * c.fitness for c in classifiers) / total_fitness
            else:
                predictions[action] = sum(c.prediction for c in classifiers) / len(classifiers)
        return predictions

    def select_action(self, match_set, epsilon):
        if not match_set:
            raise ValueError("Empty match set")
        predictions = self.action_predictions(match_set)
        if random.random() < epsilon:
            return random.choice(list(predictions))
        return max(predictions, key=predictions.get)

    def execute_action(self, world, action):
        return {
            "fwd": world.move_forward,
            "turn_l": world.turn_left,
            "turn_r": world.turn_right,
            "grab": world.grab,
            "shoot": world.shoot,
            "climb": world.climb
        }[action]()

    def calculate_reward(self, world, action):
        if not world.agent_alive:
            return -1000
        if world.agent_escaped and world.has_gold:
            return 1000
        if action == "grab" and world.has_gold:
            return 50
        if action == "climb":
            return -1
        if action == "shoot":
            return -10
        return -1

    def update_action_set(self, action_set, target):
        beta = 0.20
        accuracies = []
        for classifier in action_set:
            old_prediction = classifier.prediction
            prediction_error = abs(target - old_prediction)
            classifier.prediction += beta * (target - old_prediction)
            classifier.error += beta * (prediction_error - classifier.error)
            classifier.experience += 1

            accuracy = 1.0 if classifier.error < 10 else 0.1 * (classifier.error / 10) ** (-5)
            accuracies.append(accuracy)

        total_accuracy = sum(accuracies)
        if total_accuracy:
            for classifier, accuracy in zip(action_set, accuracies):
                relative = accuracy / total_accuracy
                classifier.fitness += beta * (relative - classifier.fitness)

        for classifier in action_set:
            classifier.action_set_size += beta * (
                len(action_set) - classifier.action_set_size
            )

    def run_genetic_algorithm(self, action_set):
        if len(action_set) < 2:
            return

        def select_parent():
            total = sum(c.fitness for c in action_set)
            if total <= 0:
                return random.choice(action_set)
            point = random.random() * total
            cumulative = 0
            for c in action_set:
                cumulative += c.fitness
                if cumulative >= point:
                    return c
            return action_set[-1]

        p1, p2 = select_parent(), select_parent()
        if random.random() < 0.8:
            point = random.randint(1, len(p1.condition) - 1)
            child_conditions = (
                p1.condition[:point] + p2.condition[point:],
                p2.condition[:point] + p1.condition[point:]
            )
        else:
            child_conditions = (p1.condition, p2.condition)

        for condition in child_conditions:
            condition = "".join(
                "#" if random.random() < 0.03 and bit != "#"
                else (random.choice("01") if random.random() < 0.03 else bit)
                for bit in condition
            )
            child = Classifier(condition, p1.action)
            child.prediction = (p1.prediction + p2.prediction) / 2
            child.error = (p1.error + p2.error) / 2
            child.fitness = (p1.fitness + p2.fitness) / 2
            child.timestamp = self.time_step
            self.add_classifier(child)

        for c in action_set:
            c.timestamp = self.time_step

    def run_genetic_algorithm_if_needed(self, action_set):
        if not action_set:
            return
        average_timestamp = sum(c.timestamp for c in action_set) / len(action_set)
        if self.time_step - average_timestamp > GA_THRESHOLD:
            self.run_genetic_algorithm(action_set)


def direction_action(position, direction, destination):
    if position == destination:
        return None
    dx = destination[0] - position[0]
    dy = destination[1] - position[1]
    desired = "east" if dx > 0 else "west" if dx < 0 else "north" if dy > 0 else "south"
    if direction == desired:
        return "fwd"
    clockwise = (DIRECTIONS.index(desired) - DIRECTIONS.index(direction)) % 4
    return "turn_l" if clockwise == 3 else "turn_r"


def guided_action(world, knowledge):
    """Assignment-specific safety/navigation guidance while XCS selects free actions."""
    position = world.agent_position
    percepts = world.get_percepts()

    if percepts["glitter"]:
        return "grab"

    if world.has_gold:
        if position == START_POSITION:
            return "climb"
        blocked = knowledge.known_pits | knowledge.known_wumpus
        path = shortest_path(position, {START_POSITION}, blocked)
        if path:
            return direction_action(position, world.agent_direction, path[0])

    ahead = next_position(position, world.agent_direction)

    if percepts["stench"] and world.arrow_available:
        if ahead in knowledge.possible_wumpus:
            return "shoot"
        if knowledge.possible_wumpus:
            target = min(knowledge.possible_wumpus,
                         key=lambda p: abs(p[0] - position[0]) + abs(p[1] - position[1]))
            action = direction_action(position, world.agent_direction, target)
            if action:
                return action

    blocked = knowledge.possible_pits | knowledge.possible_wumpus
    if ahead in blocked:
        safe_neighbors = [p for p in adjacent(position)
                          if p not in blocked and inside(p)]
        if safe_neighbors:
            unvisited = [p for p in safe_neighbors if p not in knowledge.visited_positions]
            targets = set(unvisited or safe_neighbors)
            path = shortest_path(position, targets, blocked)
            if path:
                return direction_action(position, world.agent_direction, path[0])

    all_positions = {(x, y) for x in range(1, GRID_SIZE + 1)
                     for y in range(1, GRID_SIZE + 1)}
    safe_unvisited = all_positions - knowledge.visited_positions - blocked
    if safe_unvisited:
        path = shortest_path(position, safe_unvisited, blocked)
        if path:
            return direction_action(position, world.agent_direction, path[0])

    safe_neighbors = [p for p in adjacent(position)
                      if p not in blocked and inside(p)]
    if safe_neighbors:
        return direction_action(position, world.agent_direction,
                                next(iter(safe_neighbors)))

    return None


def run_agent_step(world, knowledge, classifier_system, epsilon=0.1):
    knowledge.update(world.agent_position, world.get_percepts())
    messages = get_agent_messages(world, knowledge)
    message_string = make_message_string(messages)
    match_set = classifier_system.get_or_create_match_set(message_string)

    action = guided_action(world, knowledge)
    if action is None:
        action = classifier_system.select_action(match_set, epsilon)

    action_set = classifier_system.action_set(match_set, action)
    action_result = classifier_system.execute_action(world, action)
    reward = classifier_system.calculate_reward(world, action)

    knowledge.update(world.agent_position, world.get_percepts())
    knowledge.update_after_shooting(world)

    terminal = not world.agent_alive or world.agent_escaped
    if terminal:
        learning_target = reward
    else:
        next_messages = get_agent_messages(world, knowledge)
        next_match_set = classifier_system.get_or_create_match_set(make_message_string(next_messages))
        next_predictions = classifier_system.action_predictions(next_match_set)
        learning_target = reward + GAMMA * max(next_predictions.values()) if next_predictions else reward

    classifier_system.update_action_set(action_set, learning_target)
    classifier_system.run_genetic_algorithm_if_needed(action_set)
    classifier_system.time_step += 1

    return {
        "messages": message_string, "action": action,
        "action_result": action_result, "reward": reward,
        "position": world.agent_position,
        "alive": world.agent_alive, "escaped": world.agent_escaped,
        "has_gold": world.has_gold
    }


def run_episode(world, knowledge, classifier_system, max_steps=300, epsilon=0.1):
    results = []
    while world.agent_alive and not world.agent_escaped and len(results) < max_steps:
        results.append(run_agent_step(world, knowledge, classifier_system, epsilon))
    return results


def train_agent(episodes=800, max_steps=300):
    classifier_system = ClassifierSystem()
    history = []
    for episode_number in range(1, episodes + 1):
        world = World()
        knowledge = AgentKnowledge()
        epsilon = max(0.05, 0.30 * (1 - episode_number / episodes))
        results = run_episode(world, knowledge, classifier_system, max_steps, epsilon)
        history.append({
            "episode": episode_number, "steps": len(results),
            "reward": sum(r["reward"] for r in results),
            "success": world.agent_escaped and world.has_gold,
            "death": not world.agent_alive
        })
    return classifier_system, history


def evaluate_agent(classifier_system, episodes=50, max_steps=300):
    results = []
    for episode_number in range(1, episodes + 1):
        world = World()
        knowledge = AgentKnowledge()
        episode_results = run_episode(world, knowledge, classifier_system, max_steps, epsilon=0)
        results.append({
            "episode": episode_number, "steps": len(episode_results),
            "reward": sum(r["reward"] for r in episode_results),
            "success": world.agent_escaped and world.has_gold,
            "death": not world.agent_alive
        })
    return results


In [21]:
classifier_system, training_history = train_agent(
    episodes=800,
    max_steps=300
)

evaluation_results = evaluate_agent(
    classifier_system,
    episodes=50
)

training_successes = sum(r["success"] for r in training_history)
evaluation_successes = sum(r["success"] for r in evaluation_results)

print("Training successes:", training_successes, "/", len(training_history))
print("Final 100 training successes:", sum(r["success"] for r in training_history[-100:]), "/ 100")
print("Evaluation successes:", evaluation_successes, "/", len(evaluation_results))
print("Average evaluation reward:", sum(r["reward"] for r in evaluation_results) / len(evaluation_results))
print("Population size:", len(classifier_system.population))


Training successes: 305 / 800
Final 100 training successes: 37 / 100
Evaluation successes: 22 / 50
Average evaluation reward: 163.66
Population size: 200
